---
Extract BC Resources from publicly available RODAT data

---

PDF Source: [BCH RODAT 2013](https://www.bchydro.com/content/dam/BCHydro/customer-portal/documents/corporate/regulatory-planning-documents/integrated-resource-plans/current-plan/ror-update-appx-3-20130802.pdf)

- Load Packages

In [1]:
import fitz  # PyMuPDF
import re
import pandas as pd
from tqdm import tqdm
from pathlib import Path

In [2]:
# =====================================
# IMPORTS & CONFIG
# =====================================
import re
import fitz
import pandas as pd
from pathlib import Path
from tqdm import tqdm

pdf_path = Path("source/ror-update-appx-3-20130802.pdf")
output_dir = Path("extracts")
output_dir.mkdir(exist_ok=True)

onshore_pages = range(73, 294)  # inclusive

# =====================================
# REGEX PATTERNS (robust & multiline-tolerant)
# =====================================
option_pattern = re.compile(r"Wind[-_]?[A-Z]{2,3}\d+[A-Z]?", re.IGNORECASE)
region_pattern = re.compile(
    r"(Peace|Southern Interior|North Coast|Vancouver Island|Province[- ]?wide|British Columbia)",
    re.IGNORECASE,
)

fields = {
    "installed_capacity_MW": re.compile(
        r"Installed\s*Capacity\s*\(MW\)\s*([\d\.]+)", re.IGNORECASE
    ),
    "dependable_capacity_MW": re.compile(
        r"Dependable\s*Capacity\s*\(MW\)\s*([\d\.]+)", re.IGNORECASE
    ),
    "avg_annual_energy_GWh": re.compile(
        r"Average\s*Annual\s*Energy\s*\(GWh/year\)\s*([\d\.]+)", re.IGNORECASE
    ),
    "annual_firm_energy_GWh": re.compile(
        r"Annual\s*Firm\s*Energy\s*\(GWh/year\)\s*([\d\.]+)", re.IGNORECASE
    ),
    # ✅ fixed: multiline-tolerant Unit Energy Cost pattern
    "unit_energy_cost_$/MWh": re.compile(
        r"Unit\s*Energy\s*Cost[\s\S]{0,60}?\$?/?MWh[\s\S]{0,40}?\$?\s*([\d]{2,3}(?:\.\d+)?)",
        re.IGNORECASE,
    ),


    "capital_cost_$/kW": re.compile(
        r"Capital\s*Cost\s*\(\$?/?kW\)\s*([\d\.]+)", re.IGNORECASE
    ),
    "project_life_years": re.compile(
        r"Project\s*Life\s*\(Years\)\s*([\d\.]+)", re.IGNORECASE
    ),
    "lead_time_years": re.compile(
        r"Project\s*Lead\s*Time\s*\(Years\)\s*([\d\.]+)", re.IGNORECASE
    ),
    "effective_load_carrying_MW": re.compile(
        r"Effective\s*Load\s*Carrying\s*Capability\s*\(MW\)\s*([\d\.]+)",
        re.IGNORECASE,
    ),
}

# =====================================
# EXTRACTION
# =====================================
records = []
doc = fitz.open(pdf_path)
current_option, current_region = None, None

for page_num in tqdm(onshore_pages, desc="Extracting Onshore Wind"):
    text = doc.load_page(page_num - 1).get_text("text")
    text = re.sub(r"\s+", " ", text)  # normalize whitespace

    opt_match = option_pattern.search(text)
    if opt_match:
        current_option = opt_match.group(0)

    region_match = region_pattern.search(text)
    if region_match:
        current_region = region_match.group(1).title()

    row = {"option_id": current_option, "region": current_region}
    found = False
    for key, pat in fields.items():
        m = pat.search(text)
        if m:
            row[key] = float(m.group(1))
            found = True

    if found and current_option:
        records.append(row)

# =====================================
# DATAFRAME & CLEANUP
# =====================================
on_wind_df = pd.DataFrame(records).drop_duplicates(subset=["option_id"], keep="first")

# ensure all expected columns exist
for key in fields.keys():
    if key not in on_wind_df.columns:
        on_wind_df[key] = pd.NA

# clean any spurious high-cost captures
if "unit_energy_cost_$/MWh" in on_wind_df.columns:
    on_wind_df.loc[
        on_wind_df["unit_energy_cost_$/MWh"] > 500, "unit_energy_cost_$/MWh"
    ] = pd.NA

# derive region code
on_wind_df["region_code"] = on_wind_df["option_id"].str.extract(r"Wind_([A-Z]{2})")

# save site-level data
on_wind_df.to_csv(output_dir / "BC_onshore_wind_sites.csv", index=False)

# =====================================
# REGION SUMMARY WITH RANGES
# =====================================
summary = (
    on_wind_df.groupby("region_code")
    .agg(
        site_count=("option_id", "count"),
        capacity_range_MW=(
            "installed_capacity_MW",
            lambda x: f"{x.min():.1f}–{x.max():.1f}",
        ),
        total_capacity_MW=("installed_capacity_MW", "sum"),
        cost_range_CADMWh=(
            "unit_energy_cost_$/MWh",
            lambda x: f"{x.min():.0f}–{x.max():.0f}"
            if x.notna().any()
            else "N/A",
        ),
        avg_unit_cost_CADMWh=("unit_energy_cost_$/MWh", "mean"),
        avg_project_life_yrs=("project_life_years", "mean"),
        avg_lead_time_yrs=("lead_time_years", "mean"),
    )
    .reset_index()
)

summary.to_csv(output_dir / "BC_onshore_wind_region_summary.csv", index=False)

print("✅ Extracted", len(on_wind_df), "onshore wind options")
print("✅ Saved:", output_dir / "BC_onshore_wind_sites.csv")
print("✅ Saved:", output_dir / "BC_onshore_wind_region_summary.csv")
summary


Extracting Onshore Wind:   0%|          | 0/221 [00:00<?, ?it/s]

Extracting Onshore Wind: 100%|██████████| 221/221 [00:00<00:00, 1059.52it/s]

✅ Extracted 110 onshore wind options
✅ Saved: extracts/BC_onshore_wind_sites.csv
✅ Saved: extracts/BC_onshore_wind_region_summary.csv


,region_code,site_count,capacity_range_MW,total_capacity_MW,cost_range_CADMWh,avg_unit_cost_CADMWh,avg_project_life_yrs,avg_lead_time_yrs
0,BC,18,103.5–289.8,2950.9,100–100,100.0,20.0,5.0
1,NC,10,75.9–561.2,2152.8,100–100,100.0,20.0,5.0
2,PC,42,34.5–351.9,5214.9,100–100,100.0,20.0,5.0
3,SI,29,34.5–662.4,4105.5,100–100,100.0,20.0,5.0
4,VI,11,34.5–255.3,1014.3,100–100,100.0,20.0,5.0


In [3]:
# =====================================
# REGION SUMMARY WITH UNIT COST RANGE
# =====================================
summary = (
    on_wind_df.groupby("region_code")
    .agg(
        site_count=("option_id", "count"),
        capacity_range_MW=("installed_capacity_MW",
                           lambda x: f"{x.min():.1f}–{x.max():.1f}"),
        total_capacity_MW=("installed_capacity_MW", "sum"),
        min_unit_cost_CADMWh=("unit_energy_cost_$/MWh", "min"),
        max_unit_cost_CADMWh=("unit_energy_cost_$/MWh", "max"),
        avg_unit_cost_CADMWh=("unit_energy_cost_$/MWh", "mean"),
        avg_project_life_yrs=("project_life_years", "mean"),
        avg_lead_time_yrs=("lead_time_years", "mean"),
    )
    .reset_index()
)

# =====================================
#  ADD COST RANGE (formatted string)
# =====================================
summary["unit_cost_range_CADMWh"] = summary.apply(
    lambda row: f"{row['min_unit_cost_CADMWh']:.0f}–{row['max_unit_cost_CADMWh']:.0f}"
    if pd.notna(row["min_unit_cost_CADMWh"]) and pd.notna(row["max_unit_cost_CADMWh"])
    else "N/A",
    axis=1,
)

# Keep only the final columns you want
summary = summary[
    [
        "region_code",
        "site_count",
        "capacity_range_MW",
        "total_capacity_MW",
        "unit_cost_range_CADMWh",
        "avg_unit_cost_CADMWh",
        "avg_project_life_yrs",
        "avg_lead_time_yrs",
    ]
]

# Export summary CSV
summary.to_csv(output_dir / "BC_onshore_wind_region_summary.csv", index=False)
summary


,region_code,site_count,capacity_range_MW,total_capacity_MW,unit_cost_range_CADMWh,avg_unit_cost_CADMWh,avg_project_life_yrs,avg_lead_time_yrs
0,BC,18,103.5–289.8,2950.9,100–100,100.0,20.0,5.0
1,NC,10,75.9–561.2,2152.8,100–100,100.0,20.0,5.0
2,PC,42,34.5–351.9,5214.9,100–100,100.0,20.0,5.0
3,SI,29,34.5–662.4,4105.5,100–100,100.0,20.0,5.0
4,VI,11,34.5–255.3,1014.3,100–100,100.0,20.0,5.0
